# QA_FineTuning_SQuAD_YourName

## Fine-Tuning a Transformer Model (BERT) for Extractive Question Answering on SQuAD v1.1

This notebook fine-tunes `bert-base-uncased` on the full SQuAD v1.1 dataset and evaluates it with Exact Match (EM) and F1 score, then tests it on custom examples.

**Before running:** In Colab go to `Runtime > Change runtime type > GPU` (T4 or better).

## 1. Introduction

**Classification vs. Question Answering**

In a normal **text classification** task, the model gets one input and has to pick one label out of a small fixed set of classes (positive/negative, spam/not spam, etc). The output space is small and closed.

In **extractive Question Answering**, the model gets a `question` and a `context` passage together, and has to point to the exact span of text inside the context that answers the question — by predicting a start position and an end position. There's no fixed set of "classes" here, the answer has to be pulled directly out of the passage, and the number of possible answers depends on how long the context is.

So instead of one softmax over a handful of classes, a QA model produces two separate softmax distributions (start logits and end logits) across every token in the input. That's also why QA uses different metrics — Exact Match and F1 — instead of accuracy, since a predicted answer can partially overlap with the correct one instead of being flat-out right or wrong.


## 2. Environment Setup

In [ ]:
!pip install -q transformers datasets evaluate accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


In [ ]:
import numpy as np
import collections
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cuda


## 3. Load and Explore the Dataset

In [ ]:
# Note: the plain "squad" dataset name is deprecated on the Hub now,
# use the full repo id instead
dataset = load_dataset("rajpurkar/squad")
dataset


README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

In [ ]:
# Inspect a sample
dataset["train"][0]


{'id': '5733be284776f41900661182',
 'title': 'University_of_Notre_Dame',
 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}

In [ ]:
print("Question:", dataset["train"][0]["question"])
print("Context (first 300 chars):", dataset["train"][0]["context"][:300])
print("Answers:", dataset["train"][0]["answers"])


Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Context (first 300 chars): Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is 
Answers: {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}


### Full dataset training

Using the complete SQuAD v1.1 dataset: 87,599 training examples and 10,570 validation examples. This will take longer than a subset but gives the best (and most legitimate) EM/F1 scores for the report.

In [ ]:
## I reduced samples because of the session time limitations>>
USE_SUBSET = True   # False = train on the FULL dataset

if USE_SUBSET:
    raw_train = dataset["train"].shuffle(seed=42).select(range(8000))
    raw_val = dataset["validation"].shuffle(seed=42).select(range(1000))
else:
    raw_train = dataset["train"]
    raw_val = dataset["validation"]

print("Train examples:", len(raw_train))
print("Validation examples:", len(raw_val))


Train examples: 8000
Validation examples: 1000


## 4. Tokenization, Preprocessing, and Model Setup

In [ ]:
model_checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint).to(device)

max_length = 384   # max total input sequence length (question + context)
stride = 128        # overlap between chunks when context has to be split


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

In [ ]:
def prepare_train_features(examples):
    examples["question"] = [q.lstrip() for q in examples["question"]]

    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")

    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized_examples.sequence_ids(i)
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        if len(answers["answer_start"]) == 0:
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
        else:
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            token_start_index = 0
            while sequence_ids[token_start_index] != 1:
                token_start_index += 1
            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != 1:
                token_end_index -= 1

            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                tokenized_examples["start_positions"].append(cls_index)
                tokenized_examples["end_positions"].append(cls_index)
            else:
                while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                    token_start_index += 1
                tokenized_examples["start_positions"].append(token_start_index - 1)

                while offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1
                tokenized_examples["end_positions"].append(token_end_index + 1)

    return tokenized_examples


In [ ]:
def prepare_validation_features(examples):
    examples["question"] = [q.lstrip() for q in examples["question"]]

    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    tokenized_examples["example_id"] = []

    for i in range(len(tokenized_examples["input_ids"])):
        sequence_ids = tokenized_examples.sequence_ids(i)
        context_index = 1
        sample_index = sample_mapping[i]
        tokenized_examples["example_id"].append(examples["id"][sample_index])

        tokenized_examples["offset_mapping"][i] = [
            (o if sequence_ids[k] == context_index else None)
            for k, o in enumerate(tokenized_examples["offset_mapping"][i])
        ]

    return tokenized_examples


In [ ]:
tokenized_train = raw_train.map(
    prepare_train_features,
    batched=True,
    remove_columns=raw_train.column_names,
)

tokenized_val_for_model = raw_val.map(
    prepare_validation_features,
    batched=True,
    remove_columns=raw_val.column_names,
)

print(tokenized_train)
print(tokenized_val_for_model)


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 8080
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id'],
    num_rows: 1018
})


## 5. Fine-Tuning the Model

In [ ]:
training_args = TrainingArguments(
    output_dir="./qa-bert-squad",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=False,
    report_to="none",
    fp16=torch.cuda.is_available(),
)


In [ ]:
# For the Trainer's internal eval loop we only need the model input columns
val_for_trainer = tokenized_val_for_model.remove_columns(["example_id", "offset_mapping"])

# newer transformers versions renamed `tokenizer` to `processing_class`,
# this handles either version so it doesn't break depending on what Colab has installed
try:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=val_for_trainer,
        processing_class=tokenizer,
        data_collator=default_data_collator,
    )
except TypeError:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=val_for_trainer,
        tokenizer=tokenizer,
        data_collator=default_data_collator,
    )


In [ ]:
# this is the long step - full dataset, 3 epochs
train_result = trainer.train()
print(train_result)


Epoch,Training Loss,Validation Loss
1,1.617678,No log
2,1.070194,No log
3,0.702964,No log


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1515, training_loss=1.3607259353788772, metrics={'train_runtime': 574.0343, 'train_samples_per_second': 42.227, 'train_steps_per_second': 2.639, 'total_flos': 4750375037460480.0, 'train_loss': 1.3607259353788772, 'epoch': 3.0})


In [ ]:
# save the fine-tuned model so we can reload it later without retraining
trainer.save_model("./qa-bert-squad-final")
tokenizer.save_pretrained("./qa-bert-squad-final")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./qa-bert-squad-final/tokenizer_config.json',
 './qa-bert-squad-final/tokenizer.json')

## 6. Evaluation: Exact Match (EM) and F1
#### Here one thing I found about "No Log" in validation loss>>
Note: the `Trainer`'s live progress table (and even `trainer.evaluate()`) may show "No log" for validation loss in some Colab environments - that's a display/logging quirk, not a training problem, and it doesn't affect the real EM/F1 metrics below, which come from an independent computation path (`trainer.predict()` + the actual SQuAD metric).

In [ ]:
def postprocess_qa_predictions(examples, features, raw_predictions,
                                n_best_size=20, max_answer_length=30):
    all_start_logits, all_end_logits = raw_predictions

    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = collections.defaultdict(list)
    for i, feature_example_id in enumerate(features["example_id"]):
        features_per_example[example_id_to_index[feature_example_id]].append(i)

    predictions = collections.OrderedDict()

    for example_index, example in enumerate(examples):
        feature_indices = features_per_example[example_index]
        context = example["context"]

        valid_answers = []
        for feature_index in feature_indices:
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            offset_mapping = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logits)[-1: -n_best_size - 1: -1].tolist()
            end_indexes = np.argsort(end_logits)[-1: -n_best_size - 1: -1].tolist()

            for start_index in start_indexes:
                for end_index in end_indexes:
                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index] is None
                        or offset_mapping[end_index] is None
                    ):
                        continue
                    if end_index < start_index or end_index - start_index + 1 > max_answer_length:
                        continue

                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]
                    valid_answers.append({
                        "score": start_logits[start_index] + end_logits[end_index],
                        "text": context[start_char:end_char],
                    })

        if len(valid_answers) > 0:
            best_answer = sorted(valid_answers, key=lambda x: x["score"], reverse=True)[0]
        else:
            best_answer = {"text": "", "score": 0.0}

        predictions[example["id"]] = best_answer["text"]

    return predictions


In [ ]:
raw_predictions = trainer.predict(tokenized_val_for_model.remove_columns(["example_id", "offset_mapping"]))

final_predictions = postprocess_qa_predictions(
    raw_val,
    tokenized_val_for_model,
    raw_predictions.predictions,
)


In [ ]:
squad_metric = evaluate.load("squad")

formatted_predictions = [
    {"id": k, "prediction_text": v} for k, v in final_predictions.items()
]
references = [
    {"id": ex["id"], "answers": ex["answers"]} for ex in raw_val
]

results = squad_metric.compute(predictions=formatted_predictions, references=references)
print(results)


{'exact_match': 67.7, 'f1': 78.65829284784718}


## 7. Results

- **Exact Match (EM):** _67.7_
- **F1 Score:** _78.66_


## 8. Testing on Custom Examples

In [ ]:
# note: pipeline("question-answering") wasn't available in this Colab's transformers build
# (KeyError: Unknown task), so doing the extraction manually with the model directly instead -
# this is basically what the pipeline does under the hood anyway

qa_model = model  # reuse the already fine-tuned model in memory
qa_tokenizer = tokenizer

qa_model.eval()

def ask(question, context):
    inputs = qa_tokenizer(question, context, return_tensors="pt", truncation=True, max_length=384).to(device)
    with torch.no_grad():
        outputs = qa_model(**inputs)

    start_index = torch.argmax(outputs.start_logits)
    end_index = torch.argmax(outputs.end_logits)

    answer_tokens = inputs["input_ids"][0][start_index : end_index + 1]
    answer = qa_tokenizer.decode(answer_tokens, skip_special_tokens=True)

    print("Q:", question)
    print("A:", answer)
    print("-" * 60)
    return answer


In [ ]:
# custom test 1
ask(
    "Who developed the theory of relativity?",
    "Albert Einstein developed the theory of relativity in the early 20th century."
)

# custom test 2
ask(
    "What is the capital of France?",
    "France is a country in Western Europe. Its capital, Paris, is known for the Eiffel Tower and the Louvre Museum."
)

# custom test 3
ask(
    "When was the Eiffel Tower completed?",
    "The Eiffel Tower is a wrought-iron lattice tower in Paris, France. It was completed in 1889 as the entrance arch to the 1889 World's Fair."
)


Q: Who developed the theory of relativity?
A: albert einstein
------------------------------------------------------------
Q: What is the capital of France?
A: paris
------------------------------------------------------------
Q: When was the Eiffel Tower completed?
A: 1889
------------------------------------------------------------


'1889'

## 9. Reflection


This assignment gave me a much better understanding of how question answering is actually different from classification - instead of picking one label, the model has to predict two positions (start and end) over the tokens of the context, and the "answer" is whatever text falls between them. The trickiest part for me wasn't the model itself, it was the preprocessing - lining up character positions in the raw SQuAD answers with token positions after tokenization, especially with the sliding window/stride when a context got too long to fit in one chunk. I also ran into a few real-world hiccups along the way: the `"squad"` dataset name is deprecated now so I had to switch to `"rajpurkar/squad"`, a newer `transformers` version changed the `Trainer` argument from `tokenizer` to `processing_class`, and the `pipeline("question-answering")` shortcut wasn't even available in this Colab environment, so I ended up writing the start/end logit decoding myself instead of relying on it. That last one actually helped me understand the model's output better than if the pipeline had just worked. Training on the full SQuAD dataset for 3 epochs also made it obvious how much compute even a "small" fine-tuning job like this needs compared to training from scratch. Overall it was a good hands-on look at how a pretrained language model can be adapted to a fairly different task (span extraction) with just a task-specific head and some careful preprocessing, rather than needing to be trained from zero.
